# 🧪 Lab 5 — Praktikum Computer Vision (dari Nol, Menuju CNN)> Pendamping materi Bab 5. Prinsip sama dengan Lab 4: **CNN dibangun dan> dihitung dari nol dulu (numpy)** — konvolusi, pooling, augmentasi — supaya> `Conv2D` tidak pernah jadi kotak hitam. Semua gambar kecil (8x8) agar cepat.| Bagian | Topik | Waktu (menit) ||---|---|---|| 1 | Gambar sebagai array & konvolusi manual | 20 || 2 | Filter klasik: tepi, blur, sharpen | 15 || 3 | Padding, stride, dan rumus output | 15 || 4 | Konvolusi dengan `sliding_window_view` + backward | 25 || 5 | Max-pool 2x2: forward & backward (argmax) | 20 || 6 | CNNMini: Conv → ReLU → Pool → Flatten → Dense | 25 || 7 | Augmentasi data (flip, noise, shift) | 20 || 8 | Weight sharing vs MLP: uji invarian translasi | 25 || 9 | Diagnosa overfitting & kurva train/val | 20 || 10 | Transfer learning (konsep) & menuju Keras | 20 |Setiap latihan punya cell **✅ Cek** — jalankan, kalau ✅ berarti pemahamanmu benar.

## Bagian 1 — Gambar sebagai Array & Konvolusi ManualGambar grayscale = array `(H, W)` bernilai 0–255 (atau 0–1). RGB = `(H, W, 3)`.Semua "keajaiban" CV dimulai dari hal sesederhana ini: **angka**.Konvolusi = geser filter kecil (misal 3x3) di atas gambar, tiap posisimenghasilkan 1 angka = jumlah hasil perkalian elemen-samenjadi (*dot product*).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gambar "seni abstrak" 6x6: kotak terang di latar gelap
img = np.zeros((6, 6))
img[1:4, 2:5] = 1.0
print(img)
plt.imshow(img, cmap='gray', vmin=0, vmax=1)
plt.title('Gambar 6x6 = array angka'); plt.colorbar(); plt.show()


In [ ]:
def konvolusi_manual(img, kernel):
    """Konvolusi 'valid' (tanpa padding): output (H-h+1, W-w+1).
    Tiap posisi output = dot product filter dengan jendela gambar.
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# k = np.array([[1., 0.], [0., -1.]])
# print(konvolusi_manual(np.eye(3), k))


In [ ]:
# ✅ Cek latihan 1.1
k = np.array([[1., 0.], [0., -1.]])
img9 = np.arange(9, dtype=float).reshape(3, 3)
out = konvolusi_manual(img9, k)
assert out.shape == (2, 2), 'output valid: (3-2+1, 3-2+1)'
# out[i,j] = img[i,j] - img[i+1,j+1]  →  semua elemen = 0-4 = -4
assert np.allclose(out, -4 * np.ones((2, 2))), (
    f'hitung manual dulu: out[0,0] = img[0,0] - img[1,1] = {out[0, 0]}')
# sifat linear: konvolusi (a*img1 + b*img2) = a*conv(img1) + b*conv(img2)
img_a, img_b = np.random.rand(5, 5), np.random.rand(5, 5)
assert np.allclose(konvolusi_manual(2 * img_a + 3 * img_b, k),
                   2 * konvolusi_manual(img_a, k) + 3 * konvolusi_manual(img_b, k))
# kernel nol -> output nol
assert np.allclose(konvolusi_manual(img_a, np.zeros((2, 2))), 0)
print('✅ Latihan 1.1 benar!')


## Bagian 2 — Filter Klasik: Tepi, Blur, SharpenSebelum ada "filter yang dipelajari", ada filter yang dirancang tangan.Ini bukan sejarah mati — layer awal CNN terlatih otomatis menemukan filteryang mirip-mirip filter di bawah. Lihat apa yang tiap filter "perhatikan".

In [ ]:
filter_tepi_v  = np.array([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]])  # tepi vertikal
filter_tepi_h  = filter_tepi_v.T                                           # tepi horizontal
filter_blur    = np.ones((3, 3)) / 9.0                                     # rata-rata lokal
filter_sharpen = np.array([[0., -1., 0.], [-1., 5., -1.], [0., -1., 0.]])

# Gambar uji: kotak terang di atas latar gelap (pakai hasil latihan 1.1)
test_img = np.zeros((12, 12)); test_img[3:9, 3:9] = 1.0

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (nama, f) in zip(axes, [('asli', None), ('tepi v', filter_tepi_v),
                                ('tepi h', filter_tepi_h),
                                ('blur', filter_blur), ('sharpen', filter_sharpen)]):
    hasil = test_img if f is None else konvolusi_manual(test_img, f)
    ax.imshow(hasil, cmap='gray'); ax.set_title(nama); ax.axis('off')
plt.suptitle('Satu gambar, empat filter, empat "perhatian" berbeda')
plt.show()


**Pertanyaan pemahaman (jangan skip!):**1. Filter tepi vertikal membuat bagian mana jadi terang — sisi kiri kotak,   sisi kanan, atau keduanya? (Jalankan dulu, jawab dari hasil.)2. Kenapa hasil tepi-v dan tepi-h "saling melengkapi"?3. Di CNN, jumlah filter per layer = jumlah "jenis pola" yang dicari sekaligus.   `Conv2D(32, 3)` artinya apa dalam bahasa ini?

## Bagian 3 — Padding, Stride, dan Rumus OutputTiga knob yang menentukan ukuran output konvolusi:```
out = floor((H + 2*pad − k) / stride) + 1
```
- `pad` = nol yang ditempel di pinggir (jaga ukuran & kasih konteks tepi)
- `stride` = langkah geser (2 = output setengah ukuran, alternatif pooling)
- `padding='same'` di Keras = pad dipilih otomatis agar output = input (stride 1)

In [ ]:
def pad_zero(img, pad):
    """Keliling gambar dengan pad lapis nol. pad=0 -> img apa adanya."""
    # TODO
    raise NotImplementedError

def ukuran_output(H, k, pad=0, stride=1):
    """Rumus ukuran output konvolusi."""
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# print(pad_zero(np.ones((2, 2)), 1).shape)        # (4, 4)
# print([ukuran_output(32, 3, p, s) for p, s in [(0,1),(1,1),(0,2)]])  # [30, 32, 15]


In [ ]:
# ✅ Cek latihan 3.1
p1 = pad_zero(np.ones((2, 2)), 1)
assert p1.shape == (4, 4)
assert np.all(p1[1:3, 1:3] == 1) and np.all(p1[0] == 0) and np.all(p1[:, -1] == 0)
assert pad_zero(np.ones((3, 3)), 0).shape == (3, 3), 'pad=0 identitas'
p2 = pad_zero(np.array([[5.]]), 2)
assert p2.shape == (5, 5) and p2[2, 2] == 5 and np.all(p2[0, :] == 0)
assert ukuran_output(32, 3, 0, 1) == 30
assert ukuran_output(32, 3, 1, 1) == 32, "pad=1 + k=3 (ganjil) = 'same'"
assert ukuran_output(32, 3, 0, 2) == 15, 'floor((32-3)/2)+1'
assert ukuran_output(8, 5, 0, 1) == 4
assert ukuran_output(28, 3, 1, 2) == 14
print('✅ Latihan 3.1 benar!')


## Bagian 4 — Konvolusi Efisien + Backward (Gradient Check!)Loop bersarang di Bagian 1 jujur tapi lambat. Cara produksi: jadikan semuajendela jadi satu batch lalu `tensordot` — di numpy pakai`sliding_window_view`. Ini *im2col* yang dipakai library sungguhan.Lalu bagian paling penting: **backward**. Gradien konvolusi punya bentuksimetris yang indah (dK = konvolusi input dengan dOut; dX = konvolusi penuhdOut dengan kernel yang dibalik). Kita buktikan dengan **gradient check**numerik — alat debugging yang dipakai sungguhan di industri.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def conv2d_forward(img, K, pad=0, stride=1):
    """Konvolusi cross-correlation: img (H, W), K (kh, kw) -> out (Ho, Wo).
    img dulu di-pad, lalu jendela (kh, kw) digeser sebesar stride,
    tiap jendela di-dot dengan K (vektorisasi penuh).
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# rng = np.random.default_rng(0)
# a, k4 = rng.normal(size=(5, 5)), rng.normal(size=(3, 3))
# print(conv2d_forward(a, k4).shape)      # (3, 3)
# print(conv2d_forward(a, k4, pad=1).shape)  # (5, 5)


In [ ]:
# ✅ Cek latihan 4.1
rng = np.random.default_rng(0)
a5, k3 = rng.normal(size=(5, 5)), rng.normal(size=(3, 3))
assert conv2d_forward(a5, k3).shape == (3, 3)
assert conv2d_forward(a5, k3, pad=1).shape == (5, 5)
assert conv2d_forward(a5, np.ones((3, 3)) / 9).shape == (3, 3)
# hasil = konvolusi manual Bagian 1 (loop) — kecuali pad>0 (di luar jangkauan loop)
assert np.allclose(conv2d_forward(a5, k3), konvolusi_manual(a5, k3))
# stride 2 = ambil tiap elemen ke-2 dari hasil stride 1
o1 = conv2d_forward(a5, k3, stride=1)
o2 = conv2d_forward(a5, k3, stride=2)
assert np.allclose(o2, o1[::2, ::2]), 'stride 2 = sampling tiap-2 posisi'
print('✅ Latihan 4.1 benar!')


In [ ]:
def conv2d_backward(dOut, img, K, pad=0, stride=1):
    """Gradien loss thd K dan img.
    dK: (kh, kw)  = tiap elemen K dipakai di semua posisi -> jumlahkan kontribusi.
    dX: (H, W)    = tiap piksel menerima gradien dari SEMUA jendela yang memuatnya
                    (full-convolution dengan kernel dibalik 180 derajat).
    Return (dK, dX).
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# dOut = rng.normal(size=(3, 3))
# dK, dX = conv2d_backward(dOut, a5, k3)
# print(dK.shape, dX.shape)   # (3, 3) (5, 5)


In [ ]:
# ✅ Cek latihan 4.2 — GRADIENT CHECK numerik (bagian terpenting lab ini!)
def grad_check(fn, x, eps=1e-6, tol=1e-5):
    """Bandingkan gradien analitik vs selisih-hingga tengah."""
    x = np.asarray(x, dtype=float)
    g_num = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        orig = x[idx]
        x[idx] = orig + eps; fp = fn()
        x[idx] = orig - eps; fm = fn()
        x[idx] = orig
        g_num[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g_num

rng = np.random.default_rng(2)
img = rng.normal(size=(6, 6))
K = rng.normal(size=(3, 3))
dOut = rng.normal(size=(4, 4))
loss = lambda: float(np.sum(conv2d_forward(img, K) * dOut))
dK, dX = conv2d_backward(dOut, img, K)
assert dK.shape == (3, 3) and dX.shape == (6, 6)
gK_num = grad_check(loss, K)
gX_num = grad_check(loss, img)
rel = lambda a, b: np.max(np.abs(a - b) / (np.abs(a) + np.abs(b) + 1e-12))
assert rel(dK, gK_num) < 1e-5, f'dK meleset: {rel(dK, gK_num)}'
assert rel(dX, gX_num) < 1e-5, f'dX meleset: {rel(dX, gX_num)}'
# dengan padding: uji gradien lengkapnya secara mandiri (dOut 6x6 = output pad=1)
dOut6 = rng.normal(size=(6, 6))
loss_p = lambda: float(np.sum(conv2d_forward(img, K, pad=1) * dOut6))
dKp, dXp = conv2d_backward(dOut6, img, K, pad=1)
assert dKp.shape == (3, 3) and dXp.shape == (6, 6)
assert rel(dKp, grad_check(loss_p, K)) < 1e-5, 'dK pad=1'
assert rel(dXp, grad_check(loss_p, img)) < 1e-5, 'dX pad=1'
print('✅ Latihan 4.2 benar! dK dan dX terverifikasi numerik (pad=0 & pad=1).')


## Bagian 5 — Max-Pool 2x2: Forward & BackwardPooling = "ringkas tiap jendela 2x2 jadi 1 angka". Max-pool menyisakanfitur terkuat → invarian posisi-kecil + hemat komputasi. Backward-nya menarik:gradien hanya mengalir ke **pemenang** tiap jendela (argmax). Kalau seri?Itu *subgradien* — pilih salah satu, dua-duanya benar.

In [ ]:
def maxpool2x2_forward(x):
    """x: (H, W), H & W genap -> (H/2, W/2), tiap jendela 2x2 jadi max-nya."""
    # TODO
    raise NotImplementedError

def maxpool2x2_backward(dOut, x):
    """dOut: (H/2, W/2) -> dX: (H, W); gradien hanya ke posisi argmax tiap jendela."""
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# x = np.array([[1., 2.], [3., 4.]])
# print(maxpool2x2_forward(x))                     # 4.0
# print(maxpool2x2_backward(np.array([[10.]]), x))  # gradien di posisi 4


In [ ]:
# ✅ Cek latihan 5.1
x = np.array([[1., 2.], [3., 4.]])
assert maxpool2x2_forward(x).shape == (1, 1)
assert maxpool2x2_forward(x)[0, 0] == 4.0
xb = np.arange(16, dtype=float).reshape(4, 4)
assert maxpool2x2_forward(xb).shape == (2, 2)
assert np.allclose(maxpool2x2_forward(xb), [[5., 7.], [13., 15.]])
# backward: gradien hanya ke pemenang
dx = maxpool2x2_backward(np.array([[10.]]), x)
assert dx.shape == (2, 2)
assert dx[1, 1] == 10.0 and dx.sum() == 10.0, 'gradien seratus persen ke argmax'
dx2 = maxpool2x2_backward(np.ones((2, 2)), xb)
assert dx2.sum() == 4.0 and dx2[1, 1] == 1.0 and dx2[1, 3] == 1.0
# masking non-pemenang jadi 0 tidak mengubah hasil pool (data non-negatif)
assert np.allclose(maxpool2x2_forward(xb * dx2), maxpool2x2_forward(xb))
# subgradien: nilai seri — tetap total gradien yang benar
xt = np.ones((2, 2))
dxt = maxpool2x2_backward(np.array([[7.]]), xt)
assert dxt.sum() == 7.0, 'seri: salah satu pemenang menerima semua'
print('✅ Latihan 5.1 benar!')


## Bagian 6 — CNNMini: Merakit Layer Jadi ModelSekarang rakit: **Conv → ReLU → Pool → Flatten → Dense**. Tiap layer punyakontrak yang sama: `forward(x)` dan `backward(dout)`. Model = daftar layeryang diteruskan maju-mundur — persis cara `keras.Sequential` bekerja di dalam.Dataset: **Bars** 8x8 (garis horizontal vs vertikal + noise) — split 420/90/90,seed 42, sama persis dengan project starter bab ini.

In [ ]:
# Dataset Bars: latar noise + satu garis terang horizontal (label 0)
# atau vertikal (label 1), plus titik noise kasar. IDENTIK dgn project starter.
def make_bars(n=600, img=8, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 0.05, (n, img, img))
    y = np.zeros(n, dtype=int)
    for i in range(n):
        p = rng.integers(0, img)
        if rng.random() < 0.5:
            X[i, p, :] += 1.0          # garis horizontal
        else:
            y[i] = 1
            X[i, :, p] += 1.0          # garis vertikal
        for _ in range(4):             # titik noise kasar
            X[i, rng.integers(img), rng.integers(img)] += rng.choice([-1., 1.]) * 0.8
    return X, y

X_all, y_all = make_bars()
X_tr, y_tr = X_all[:420], y_all[:420]
X_va, y_va = X_all[420:510], y_all[420:510]
X_te, y_te = X_all[510:600], y_all[510:600]
print(f'train {X_tr.shape}, val {X_va.shape}, test {X_te.shape}')
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, xi, yi in zip(axes, X_all[:6], y_all[:6]):
    ax.imshow(xi, cmap='gray'); ax.set_title(f'y={yi}'); ax.axis('off')
plt.show()


In [ ]:
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1.0 / (1.0 + np.exp(-z))

def bce(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

class Dense:
    """Dari Bab 4: z = x @ W + b (input sudah flat)."""
    def __init__(self, n_in, n_out, rng, lr=0.1, use_bias=True):
        self.W = rng.normal(0, np.sqrt(2.0 / n_in), (n_in, n_out))
        self.b = np.zeros(n_out)
        self.lr = lr
        self.use_bias = use_bias

    def forward(self, x):
        self.x = x
        return x @ self.W + (self.b if self.use_bias else 0)

    def backward(self, dout):
        self.dW = self.x.T @ dout
        if self.use_bias:
            self.db = dout.sum(0)
        return dout @ self.W.T

    def step(self):
        self.W -= self.lr * self.dW
        if self.use_bias:
            self.b -= self.lr * self.db


### 🎯 Latihan 6.1 — Layer ReLU, MaxPool, Flatten, ConvEmpat layer dengan kontrak `forward`/`backward`/`step`. ReLU & Pool tak punyaparameter (`step` = no-op). Untuk `ConvLayer`: simpan cache yang dibutuhkanbackward, akumulasi gradien kernel **sepanjang batch** (`dK += ` per sampel).

In [ ]:
class ReLU:
    def forward(self, x):
        # TODO: simpan mask, return max(x, 0)
        raise NotImplementedError

    def backward(self, dout):
        # TODO: gradien lewat hanya di posisi x > 0
        raise NotImplementedError

    def step(self):
        pass

class Pool2x2:
    def forward(self, x):
        # TODO: batch (n, H, W) -> (n, H/2, W/2) + cache untuk backward
        raise NotImplementedError

    def backward(self, dout):
        # TODO: per sampel, panggil maxpool2x2_backward
        raise NotImplementedError

    def step(self):
        pass

class Flatten:
    def forward(self, x):
        # TODO: (n, H, W) -> (n, H*W), simpan shape
        raise NotImplementedError

    def backward(self, dout):
        # TODO: kembalikan bentuk semula
        raise NotImplementedError

    def step(self):
        pass

class ConvLayer:
    """Satu kernel (kh, kw) dipakai ke seluruh batch (weight sharing!)."""
    def __init__(self, ksize, rng, lr=0.05, pad=0):
        self.K = rng.normal(0, np.sqrt(2.0 / (ksize * ksize)), (ksize, ksize))
        self.lr = lr; self.pad = pad

    def forward(self, X):
        # TODO: conv2d_forward per sampel; simpan X untuk backward
        raise NotImplementedError

    def backward(self, dOut):
        # TODO: dK = jumlah gradien seluruh batch; dX per sampel
        raise NotImplementedError

    def step(self):
        self.K -= self.lr * self.dK


In [ ]:
# ✅ Cek latihan 6.1 — gradient check per layer + end-to-end
rng = np.random.default_rng(5)
Xc = rng.normal(size=(3, 6, 6))
dc = rng.normal(size=(3, 4, 4))
rel = lambda a, b: np.max(np.abs(a - b) / (np.abs(a) + np.abs(b) + 1e-12))

def grad_check_layer(layer, X, dUp, attr):
    """Cek gradien parameter layer attr ('K' atau 'W') via selisih-hingga."""
    P = getattr(layer, attr)
    g_num = np.zeros_like(P)
    it = np.nditer(P, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        orig = P[idx]
        P[idx] = orig + 1e-6; fp = float(np.sum(layer.forward(X) * dUp))
        P[idx] = orig - 1e-6; fm = float(np.sum(layer.forward(X) * dUp))
        P[idx] = orig
        g_num[idx] = (fp - fm) / 2e-6
        it.iternext()
    layer.forward(X)
    layer.backward(dUp)
    g_an = getattr(layer, 'd' + attr)
    return rel(g_an, g_num)

cl = ConvLayer(3, np.random.default_rng(1), lr=0.05)
e_conv = grad_check_layer(cl, Xc, dc, 'K')
assert e_conv < 1e-5, f'ConvLayer.dK meleset: {e_conv}'
print(f'conv dK error relatif: {e_conv:.2e} ✓')

rl = ReLU(); xr = rng.normal(size=(2, 3, 3))
h = rl.forward(xr)
assert np.allclose(h, np.maximum(xr, 0))
dr = rl.backward(np.ones_like(xr))
assert np.allclose(dr, (xr > 0).astype(float))

pl = Pool2x2(); xp = np.arange(16, dtype=float).reshape(2, 2, 4)
hp = pl.forward(xp)
assert hp.shape == (2, 1, 2) and np.allclose(hp[0], [[5., 7.]])
dp = pl.backward(np.ones_like(hp))
assert dp.shape == xp.shape and dp.sum() == 4.0, 'tepat satu gradien per jendela'
assert dp[0][1, 1] == 1.0 and dp[0][1, 3] == 1.0

fl = Flatten(); xf = rng.normal(size=(4, 3, 5))
hf = fl.forward(xf)
assert hf.shape == (4, 15) and np.allclose(fl.backward(hf), xf)

print('✅ Latihan 6.1 benar! Semua layer lolos kontrak forward/backward.')


In [ ]:
# ✅ Cek latihan 6.2 — CNNMini end-to-end: harus konvergen di task Bars
def latih_cnn(Xtr, ytr, Xva, yva, epochs=60, lr=0.5, batch_size=32, seed=0):
    """Mini-batch SGD: shuffle tiap epoch, update per batch (pola Bab 4)."""
    r = np.random.default_rng(seed)
    layers = [ConvLayer(3, r, lr=lr), ReLU(), Pool2x2(), Flatten(),
              Dense(3 * 3, 1, r, lr=lr)]
    hist = []
    for ep in range(epochs):
        order = np.random.default_rng(seed * 1000 + ep).permutation(len(Xtr))
        for s in range(0, len(Xtr), batch_size):
            idx = order[s:s + batch_size]
            Xb, yb = Xtr[idx], ytr[idx]
            a = Xb
            for L in layers:
                a = L.forward(a)
            p = sigmoid(a.ravel())
            d = ((p - yb) / len(yb)).reshape(-1, 1)   # BCE + sigmoid = sederhana
            for L in reversed(layers):
                d = L.backward(d)
            for L in layers:
                L.step()
        av = Xva
        for L in layers:
            av = L.forward(av)
        pv = (sigmoid(av.ravel()) >= 0.5).astype(int)
        hist.append(float((pv == yva).mean()))
    return layers, hist

model_cnn, hist = latih_cnn(X_tr, y_tr.astype(float), X_va, y_va)
print(f'val acc terakhir: {hist[-1]:.3f} | terbaik: {max(hist):.3f}')
plt.plot(hist); plt.xlabel('epoch'); plt.ylabel('val acc')
plt.title('CNNMini belajar task Bars'); plt.show()
assert hist[-1] >= 0.9, f'val acc harus >= 0.9, dapat {hist[-1]:.3f}'
print('✅ Latihan 6.2 benar! CNN buatanmu menyelesaikan task garis.')


## Bagian 7 — Augmentasi DataAugmentasi = "memperbanyak" data dengan transformasi yang TIDAK mengubah label.Kucing menghadap kiri atau kanan tetap kucing → flip horizontal aman.Angka 6 di-flip vertikal jadi sesuatu yang bukan 6 → merusak label!Aturan emas: **augmentasi mencerminkan variasi dunia nyata yang dijamin takmengubah kelas**. Bonus: augmentasi ideal harus *invarian posisi* — shiftgambar tidak boleh mengubah label.

In [ ]:
def flip_horizontal(img):
    """Cermin kiri-kanan."""
    # TODO
    raise NotImplementedError

def flip_vertical(img):
    """Cermin atas-bawah."""
    # TODO
    raise NotImplementedError

def random_noise(img, rng, std=0.1):
    """Tambah noise Gaussian; return salinan (jangan merusak asli!)."""
    # TODO
    raise NotImplementedError

def random_shift(img, rng, max_shift=1):
    """Geser gambar (dy, dx) acak dalam [-max_shift, max_shift];
    area kosong = 0 (bukan wrap-around — piksel bekas jangan muncul di sisi lain)."""
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# g = np.arange(16, dtype=float).reshape(4, 4)
# print(flip_horizontal(g))
# print(random_shift(g, np.random.default_rng(0), 1))


In [ ]:
# ✅ Cek latihan 7.1
g = np.arange(16, dtype=float).reshape(4, 4)
assert np.allclose(flip_horizontal(g), g[:, ::-1])
assert np.allclose(flip_horizontal(g), g[:, ::-1])
assert np.allclose(flip_vertical(g), g[::-1, :])
assert np.allclose(flip_horizontal(flip_horizontal(g)), g), 'flip 2x = identitas'
assert np.allclose(flip_vertical(flip_vertical(g)), g)
g0 = g.copy()
rngn = np.random.default_rng(1)
gn = random_noise(g, rngn, std=0.1)
assert gn.shape == g.shape and np.allclose(g, g0), 'input TIDAK berubah'
assert not np.allclose(gn, g), 'ada noise'
assert np.allclose(random_noise(g, np.random.default_rng(3), std=0.0), g), 'std=0 = asli'
r2 = np.random.default_rng(2)
gs = random_shift(g, r2, 1)
assert gs.shape == g.shape
assert np.allclose(random_shift(g, np.random.default_rng(4), 0), g), 'shift 0 = identitas'
print('✅ Latihan 7.1 benar!')


In [ ]:
# Visual: satu gambar Bars, beberapa varian augmentasi
rng_aug = np.random.default_rng(7)
contoh = X_te[0]
fig, axes = plt.subplots(2, 4, figsize=(11, 5))
axes[0, 0].imshow(contoh, cmap='gray'); axes[0, 0].set_title('asli'); axes[0, 0].axis('off')
axes[1, 0].imshow(flip_horizontal(contoh), cmap='gray'); axes[1, 0].set_title('flip h'); axes[1, 0].axis('off')
axes[0, 1].imshow(random_noise(contoh, rng_aug, 0.15), cmap='gray'); axes[0, 1].set_title('noise'); axes[0, 1].axis('off')
for j in range(2, 4):
    axes[0, j].imshow(random_shift(contoh, rng_aug, 2), cmap='gray'); axes[0, j].set_title('shift'); axes[0, j].axis('off')
    axes[1, j].imshow(random_shift(random_noise(contoh, rng_aug, 0.1), rng_aug, 2), cmap='gray')
    axes[1, j].set_title('noise + shift'); axes[1, j].axis('off')
axes[1, 1].imshow(flip_vertical(contoh), cmap='gray'); axes[1, 1].set_title('flip v (label rusak!)'); axes[1, 1].axis('off')
plt.suptitle('Flip vertikal merusak label Bars: garis h jadi v!'); plt.show()
print('Label gambar di atas:', y_te[0], '(0 = garis horizontal)')


## Bagian 8 — Weight Sharing vs MLP: Uji Invarian TranslasiIni demonstrasi inti "kenapa CNN": MLP menghafal **posisi piksel**, CNNmengenali **pola di posisi mana pun** (satu filter untuk semua posisi =*weight sharing*). Ujiannya ketat: training hanya memakai garis di posisi1–6; posisi 0 dan 7 (tepi) TIDAK PERNAH muncul saat training — hanya saat ujian.Catatan desain: CNN di bawah sengaja TANPA bias di dense layer — kalau adabias, model bisa memakai jalan pintas 'kecerahan rata-rata gambar' dan menghindaritugas mendeteksi garis (shortcut learning — masalah nyata di CV!).

In [ ]:
# Dataset posisi terbatas: garis HANYA di baris/kolom 1..6 saat training
def make_bars_pos(n, img=8, positions=None, seed=0):
    rng = np.random.default_rng(seed)
    if positions is None:
        positions = list(range(img))
    X = rng.normal(0, 0.05, (n, img, img))
    y = np.zeros(n, dtype=int)
    for i in range(n):
        p = rng.choice(positions)
        if rng.random() < 0.5:
            X[i, p, :] += 1.0
        else:
            y[i] = 1
            X[i, :, p] += 1.0
        for _ in range(4):
            X[i, rng.integers(img), rng.integers(img)] += rng.choice([-1., 1.]) * 0.8
    return X, y

X_pos, y_pos = make_bars_pos(510, positions=[1, 2, 3, 4, 5, 6], seed=42)
Xtr_p, ytr_p = X_pos[:420], y_pos[:420].astype(float)
Xva_p, yva_p = X_pos[420:], y_pos[420:]
X_ekstrem, y_ekstrem = make_bars_pos(90, positions=[0, 7], seed=99)

def akurasi(model, X, y):
    a = X
    for L in model:
        a = L.forward(a)
    p = (sigmoid(a.ravel()) >= 0.5).astype(int)
    return float((p == y).mean())

def latih_mlp(Xtr, ytr, epochs=60, lr=0.5, batch_size=32, seed=0):
    """MLP: Flatten -> Dense -> ReLU -> Dense (arsitektur Bab 4, tanpa conv)."""
    r = np.random.default_rng(seed)
    layers = [Flatten(), Dense(64, 16, r, lr=lr), ReLU(), Dense(16, 1, r, lr=lr)]
    for ep in range(epochs):
        order = np.random.default_rng(seed * 1000 + ep).permutation(len(Xtr))
        for s in range(0, len(Xtr), batch_size):
            idx = order[s:s + batch_size]
            Xb, yb = Xtr[idx], ytr[idx]
            a = Xb
            for L in layers:
                a = L.forward(a)
            p = sigmoid(a.ravel())
            d = ((p - yb) / len(yb)).reshape(-1, 1)
            for L in reversed(layers):
                d = L.backward(d)
            for L in layers:
                L.step()
    return layers

def latih_cnn_nobias(Xtr, ytr, epochs=60, lr=0.5, batch_size=32, seed=0):
    """CNN pad=0 tanpa bias dense: shortcut 'kecerahan rata-rata' tertutup."""
    r = np.random.default_rng(seed)
    layers = [ConvLayer(3, r, lr=lr), ReLU(), Pool2x2(), Flatten(),
              Dense(9, 8, r, lr=lr, use_bias=False), ReLU(),
              Dense(8, 1, r, lr=lr, use_bias=False)]
    for ep in range(epochs):
        order = np.random.default_rng(seed * 1000 + ep).permutation(len(Xtr))
        for s in range(0, len(Xtr), batch_size):
            idx = order[s:s + batch_size]
            Xb, yb = Xtr[idx], ytr[idx]
            a = Xb
            for L in layers:
                a = L.forward(a)
            p = sigmoid(a.ravel())
            d = ((p - yb) / len(yb)).reshape(-1, 1)
            for L in reversed(layers):
                d = L.backward(d)
            for L in layers:
                L.step()
    return layers

model_mlp = latih_mlp(Xtr_p, ytr_p, seed=0)
model_cnn2 = latih_cnn_nobias(Xtr_p, ytr_p, seed=0)
acc_in_mlp  = akurasi(model_mlp, Xtr_p, ytr_p)
acc_out_mlp = akurasi(model_mlp, X_ekstrem, y_ekstrem)
acc_in_cnn  = akurasi(model_cnn2, Xtr_p, ytr_p)
acc_out_cnn = akurasi(model_cnn2, X_ekstrem, y_ekstrem)
print(f'MLP: in-distribution {acc_in_mlp:.3f} | posisi terlarang {acc_out_mlp:.3f}')
print(f'CNN: in-distribution {acc_in_cnn:.3f} | posisi terlarang {acc_out_cnn:.3f}')
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(X_ekstrem[0], cmap='gray'); axes[0].set_title(f'test: garis di tepi (y={y_ekstrem[0]})')
axes[1].imshow(X_ekstrem[1], cmap='gray'); axes[1].set_title(f'test: garis di tepi (y={y_ekstrem[1]})')
axes[2].bar(['MLP', 'CNN'], [acc_out_mlp, acc_out_cnn]); axes[2].set_ylim(0, 1)
axes[2].set_title('Akurasi pada posisi tak pernah dilihat')
plt.show()


### 🎯 Latihan 8.1 — Analisis HasilJawab dengan kata-kata (tulis di cell markdown di bawah):1. Bandingkan `acc_out_mlp` vs `acc_out_cnn`. Pola apa yang terlihat?2. Kenapa CNN JAUH lebih tahan di posisi terlarang padahal TIDAK PERNAH melihat   garis di posisi itu saat training? (hint: satu filter digeser ke SEMUA posisi —   deteksi garis tidak bergantung posisi). Kenapa CNN pun tidak 100%? (hint:   tanpa padding, garis di tepi hanya terlihat setengah oleh filter 3x3)3. Kalau augmentasi shift ditambahkan ke training MLP, prediksimu: akurasi   posisi terlarang naik atau tetap? Kenapa? (ingat: augmentasi = data baru   yang konsisten label-nya — bukan sihir yang bekerja pada arsitektur)

In [ ]:
# ✅ Cek latihan 8.1
assert acc_out_cnn > acc_out_mlp, \
    f'CNN harus lebih tahan shift: cnn={acc_out_cnn:.3f} vs mlp={acc_out_mlp:.3f}'
assert acc_in_mlp > 0.9 and acc_in_cnn > 0.9, 'keduanya harus kuat di in-distribution'
print(f'✅ Terbukti: MLP {acc_out_mlp:.2f} vs CNN {acc_out_cnn:.2f} pada posisi terlarang.')
print('   (Tulis analisis kata-katamu di cell markdown — angka saja tidak cukup!)')


## Bagian 9 — Diagnosa Overfitting dengan Kurva Train/ValModel besar + data sedikit = resep overfitting: train loss terus turun,val loss berbalik naik. Obat: **augmentasi** (cara CV menambah data),regularisasi, dan early stopping. Kita reproduksi di task Bars dengantraining set yang sengaja diperkecil.

In [ ]:
def latih_dengan_history(model_builder, Xtr, ytr, Xva, yva, epochs=300, lr=0.5,
                          batch_size=32, seed=2):
    model = model_builder()
    tr_hist, va_hist = [], []
    for ep in range(epochs):
        order = np.random.default_rng(seed * 1000 + ep).permutation(len(Xtr))
        for s in range(0, len(Xtr), batch_size):
            idx = order[s:s + batch_size]
            a = Xtr[idx]
            for L in model:
                a = L.forward(a)
            p = sigmoid(a.ravel())
            d = ((p - ytr[idx]) / len(idx)).reshape(-1, 1)
            for L in reversed(model):
                d = L.backward(d)
            for L in model:
                L.step()
        a = Xtr
        for L in model:
            a = L.forward(a)
        tr_hist.append(bce(ytr, sigmoid(a.ravel())))
        a = Xva
        for L in model:
            a = L.forward(a)
        va_hist.append(bce(yva, sigmoid(a.ravel())))
    return model, tr_hist, va_hist

# Data kecil: 120 sampel + hidden layer besar — resep klasik overfitting
X_kecil, y_kecil = X_tr[:120], y_tr[:120].astype(float)

def cnn_besar():
    r = np.random.default_rng(2)
    return [ConvLayer(3, r, lr=0.5), ReLU(), Pool2x2(), Flatten(),
            Dense(9, 128, r, lr=0.5), ReLU(), Dense(128, 1, r, lr=0.5)]

_, tr_hist, va_hist = latih_dengan_history(cnn_besar, X_kecil, y_kecil, X_va, y_va.astype(float))
plt.plot(tr_hist, label='train loss')
plt.plot(va_hist, label='val loss')
plt.xlabel('epoch'); plt.ylabel('BCE'); plt.legend()
plt.title('Overfitting: val loss berbalik naik sementara train menuju nol')
plt.show()
best_ep = int(np.argmin(va_hist))
print(f'val loss terbaik {min(va_hist):.4f} di epoch {best_ep}; akhir {va_hist[-1]:.4f}')
print('Keras equivalen: EarlyStopping(monitor="val_loss", patience=k, restore_best_weights=True)')
assert tr_hist[-1] < 0.05, 'train loss harus mendekati nol (model menghafal)'
assert va_hist[-1] > min(va_hist) + 0.2, 'val loss harus naik jauh setelah titik terbaik'
print('✅ Overfitting tereproduksi: train ~0, val memburuk setelah epoch terbaik.')


### 🎯 Latihan 9.1 — Logika Early StoppingLogika early stopping bisa (dan sebaiknya) diuji sebagai fungsi murni —tanpa melatih model. Identik dengan Latihan 9.1 Lab 4: kalau dulu lancar,ulang dari ingatan.

In [ ]:
def cek_early_stop(val_loss, patience):
    """Return (best_epoch, stop_epoch).
    best_epoch = index val_loss TERENDAH (yang pertama kalau seri).
    stop_epoch = epoch pertama di mana 'tidak membaik' beruntun mencapai
                 patience; kalau tidak pernah -> len(val_loss) - 1.
    'Membaik' = val_loss[i] < best_so_far - 1e-6 (best_so_far mulai inf).
    """
    # TODO
    raise NotImplementedError

# --- uji sendiri ---
# print(cek_early_stop([1.0, 0.8, 0.7, 0.71, 0.72, 0.73], patience=2))


In [ ]:
# ✅ Cek latihan 9.1
assert cek_early_stop([1.0, 0.8, 0.7, 0.71, 0.72, 0.73], 2) == (2, 4)
assert cek_early_stop([0.5, 0.4, 0.3, 0.2], 3) == (3, 3), 'selalu membaik -> tak berhenti'
assert cek_early_stop([0.3, 0.3, 0.3, 0.3], 2) == (0, 2), 'stagnan = tidak membaik'
assert cek_early_stop([0.9, 0.5, 0.6, 0.4, 0.55, 0.35], 1) == (1, 2), 'patience=1: stop di pelanggaran pertama'
assert cek_early_stop([0.7], 5) == (0, 0)
print('✅ Latihan 9.1 benar!')


## Bagian 10 — Transfer Learning (Konsep) & Menuju KerasSemua yang kamu bangun manual — conv, pool, flatten, dense, training loop —adalah isi `keras.Sequential` di bawah. Transfer learning menambah satu ide:**jangan latih dari nol kalau ada yang sudah melatih**. Filter layer awal(tepi, tekstur) bersifat umum untuk hampir semua gambar — jadi pakai ulangbobot ImageNet, ganti hanya "kepala" klasifikator. Persis pola LLM: pre-trained+ adaptasi, bukan training dari nol.```
Data sedikit (< 1000/kelas)  → freeze base, latih head saja + augmentasi agresif
Data sedang (1k–10k)         → fine-tune sebagian layer atas, learning rate kecil
Data banyak (> 10k/kelas)    → baru pertimbangkan training dari nol
```
Kenapa learning rate lebih kecil saat fine-tune? Bobot base sudah dekatsolusi bagus — langkah besar justru MERUSAK fitur yang sudah baik.

In [ ]:
# Peta numpy -> Keras: hal yang sama, 10 baris (jalankan kalau TF terpasang)
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    # 1) CNN dari nol versi Keras — padanan CNNMini Bagian 6
    model_scratch = keras.Sequential([
        layers.Input(shape=(8, 8, 1)),
        layers.Conv2D(1, 3, activation='relu'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(1, activation='sigmoid'),
    ])
    model_scratch.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    hist = model_scratch.fit(X_tr[..., None], y_tr, validation_split=0.2,
                             epochs=10, batch_size=32, verbose=0)
    print(f'Keras scratch: val acc = {hist.history["val_accuracy"][-1]:.3f}')

    # 2) Transfer learning: MobileNetV2 ImageNet, base dibekukan
    base = keras.applications.MobileNetV2(input_shape=(96, 96, 3),
                                          include_top=False, weights='imagenet',
                                          pooling='avg')
    base.trainable = False
    model_tl = keras.Sequential([base, layers.Dense(10, activation='softmax')])
    model_tl.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
    print(f'Pola transfer learning: {model_tl.count_params():,} param total,')
    print(f'  tapi hanya {sum(int(np.prod(w.shape)) for w in model_tl.trainable_weights):,} yang dilatih.')
    print('-> Itulah "freezer": fitur umum dipakai ulang, kepala khusus task dilatih.')
except ImportError:
    print('TensorFlow belum terpasang — konsepnya tetap valid:')
    print('  layerConv(Pad+Conv) ~ ConvLayer-mu; RandomFlip ~ flip_horizontal-mu;')
    print('  MaxPooling2D ~ Pool2x2-mu; base.trainable=False ~ freeze.')
    print('  Semua yang kamu butuh untuk MEMAHAMINYA sudah ada di lab ini.')


**Refleksi terakhir:**1. Jelaskan ke teman: kenapa konvolusi lebih cocok untuk gambar daripada Dense   biasa? Gunakan kata *weight sharing*, *fitur lokal*, dan *invarian translasi*.2. Gradient check membuktikan dK dan dX-mu benar. Bagian backward mana yang   masih samar — dK (akumulasi) atau dX (full-conv)? Ulangi Bagian 4 kalau perlu.3. Sebut 3 hal yang Keras berikan gratis dari yang kamu tulis manual di lab ini.

## 🏁 Penutup Lab**Yang sudah kamu bangun dari nol:** konvolusi (forward + backward yangterverifikasi gradient check), max-pool dengan backward argmax, pipelineaugmentasi yang aman-label, model CNN lengkap yang menang melawan MLPdi ujian invarian translasi, dan pemahaman transfer learning.**Lanjut:** `02_kuis_computer_vision.ipynb` → `project-starter-cnn-mini/`→ `cheatsheet-computer-vision.md`.